# song-cloner on Google Colab

Train a voice from songs you provide, then convert songs to that voice, on Colab's free GPU.
Everything is kept in a `song-cloner` folder on your Google Drive, so nothing is lost when the
Colab session ends, and a later run continues where this one stopped.

**Before you start**

1. **Runtime → Change runtime type → T4 GPU.** A TPU does not work: this runs on PyTorch with
   NVIDIA GPUs. The notebook stops with a message if there is no GPU.
2. Run the cells from top to bottom. The first run creates the folders on your Drive and tells you
   where to put your songs.

The voice and the songs you make are for your own listening. Do not publish or share them.

In [ ]:
# --- Settings: edit these, then run the cells in order ----------------------
VOICE_NAME = "myvoice"      # a short name for the voice: letters, digits, - _ .
EPOCHS = 300                # training stops early by itself when the voice stops improving
BATCH_SIZE = 8              # lower to 4 if training fails with "out of memory"
QUALITY = "high"            # "high" or "standard" (faster, a little less clean)
DRIVE_FOLDER = "/content/drive/MyDrive/song-cloner"
CODE_VERSION = "claude/high-quality-pipeline"   # which version of song-cloner to install

In [ ]:
# 1. Connect Google Drive and create the folders.
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
songs_folder = Path(DRIVE_FOLDER) / "songs" / VOICE_NAME
for folder in (songs_folder, Path(DRIVE_FOLDER) / "inbox", Path(DRIVE_FOLDER) / "covers"):
    folder.mkdir(parents=True, exist_ok=True)
print(f"Put the training songs here:   {songs_folder}")
print(f"Put songs to convert here:     {DRIVE_FOLDER}/inbox")
print(f"Converted songs will appear in {DRIVE_FOLDER}/covers")

**Training songs.** Upload 15–25 songs to the `songs/<voice name>` folder printed above (in
Google Drive: My Drive → song-cloner → songs). Choose songs where the singer is alone: no duets,
little chorus. Common audio formats work (mp3, m4a, wav, flac). The aim is 20 minutes or more of
solo singing; the notebook tells you how much it found.

In [ ]:
# 2. Install song-cloner (about 5 minutes; needed once per Colab session).
!rm -rf /content/song-cloner
!git clone --quiet --depth 1 --branch "$CODE_VERSION" https://github.com/aniryou/song-cloner /content/song-cloner
!bash /content/song-cloner/scripts/setup_gpu.sh /content/sc-env

In [ ]:
# 3. Check the GPU. This stops with an explanation if the runtime has no GPU.
import subprocess

os.environ["SONGCLONER_BUCKET_DIR"] = DRIVE_FOLDER
os.environ["SONGCLONER_WORK_DIR"] = "/content/songcloner-work"
os.environ["PYTHONUNBUFFERED"] = "1"

def run(*args):
    """Run one song-cloner step, showing its output here; stop the notebook if it fails."""
    command = ["/content/sc-env/bin/python", "-m", "songcloner.job", *args, "--require-gpu"]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise SystemExit("This step did not finish. The reason is in the output above.")

run("smoke")

## Train the voice

**Step A** strips the music, backing vocals and reverb from each song and reports how many minutes
of singing it found. Expect roughly a minute or two per song. The isolated vocals are saved to
`datasets/` on your Drive, so songs are never processed twice: if the report says there is too
little singing, add songs to the folder and run the cell again.

In [ ]:
# Step A: isolate the singer's voice from the training songs.
run("train", VOICE_NAME, "--quality", QUALITY, "--dataset-only")

**Step B** trains the voice. This is the long part: keep this browser tab open. While it trains,
the best version so far is copied to `voices/` on your Drive every two minutes, so if Colab
disconnects you still have a usable voice, and you can run this cell again later.

In [ ]:
# Step B: train.
run("train", VOICE_NAME, "--quality", QUALITY,
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE))

## Convert a song

Upload the song to the `inbox` folder on your Drive (My Drive → song-cloner → inbox), put its file
name below and run the cell. The result is saved to `covers/` and can be played here.

`OCTAVES`: use `-1` when the original singer is a woman and the voice is a man's (`1` for the
reverse); leave `0` when they are in the same range. `SEMITONES` changes the key of the whole
song, for when it sits too high or low for the voice.

In [ ]:
SONG = "example.mp3"   # a file in the inbox folder
OCTAVES = 0
SEMITONES = 0

before = set((Path(DRIVE_FOLDER) / "covers").iterdir())
run("convert", f"inbox/{SONG}", "--voice", VOICE_NAME, "--quality", QUALITY,
    "--octaves", str(OCTAVES), "--semitones", str(SEMITONES))
new = sorted(set((Path(DRIVE_FOLDER) / "covers").iterdir()) - before)
print("Saved:", *new, sep="\n  ")

In [ ]:
# Listen to the newest cover.
from IPython.display import Audio
covers = sorted((Path(DRIVE_FOLDER) / "covers").iterdir(), key=lambda p: p.stat().st_mtime)
print(covers[-1].name)
Audio(str(covers[-1]))